In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  # ImageNet mean
                         [0.229, 0.224, 0.225])   # ImageNet std
])


In [4]:
train_dataset = datasets.ImageFolder("../data/dataset_split/train", transform=transform)
val_dataset = datasets.ImageFolder("../data/dataset_split/val", transform=transform)


In [ ]:

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

for param in model.features.parameters():
    param.requires_grad = False


num_classes = len(train_dataset.classes)
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)


In [ ]:
from tqdm import tqdm
import torch

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f"\n🔁 Epoch {epoch+1}/{num_epochs}")
    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"✅ Epoch {epoch+1} Completed — Loss: {total_loss:.4f}")


    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"🎯 Validation Accuracy after Epoch {epoch+1}: {accuracy:.2f}%")



🔁 Epoch 1/5


✅ Epoch 1 Completed — Loss: 330.5378


🎯 Validation Accuracy after Epoch 1: 92.72%

🔁 Epoch 2/5


✅ Epoch 2 Completed — Loss: 155.6557


🎯 Validation Accuracy after Epoch 2: 93.95%

🔁 Epoch 3/5


✅ Epoch 3 Completed — Loss: 127.2919


🎯 Validation Accuracy after Epoch 3: 94.46%

🔁 Epoch 4/5


✅ Epoch 4 Completed — Loss: 114.5497


🎯 Validation Accuracy after Epoch 4: 94.82%

🔁 Epoch 5/5


✅ Epoch 5 Completed — Loss: 110.0571


🎯 Validation Accuracy after Epoch 5: 94.94%


In [11]:
torch.save(model.state_dict(), "../models/plant_disease_model.pth")


In [12]:
from PIL import Image

def predict_image(image_path, model, transform, class_names):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
        predicted_class = class_names[predicted.item()]
        return predicted_class


In [13]:
class_names = train_dataset.classes
prediction = predict_image("../data/dataset_split/train/Potato___Early_blight/0a79700b-f834-41f5-ae51-6ceda6f67a48___RS_Early.B 8951.JPG", model, transform, class_names)
print("Predicted Disease:", prediction)


Predicted Disease: Potato___Early_blight
